# Bronze → Silver Transformation Pipeline (Azure Synapse PySpark)

Notebook ini membaca hasil scraping harian dari ADLS Gen2 container **`bronze`** (`runs/YYYY/MM/DD/HH00/`), melakukan cleaning, casting tipe data, **concat (`unionByName`)** dengan dataset historis di container **`silver`**, serta **deduplikasi** untuk 5 sumber data:

1. `firms` — NASA FIRMS Hotspots (MODIS & VIIRS)
2. `magma` — MAGMA Indonesia (Aktivitas Gunung Api)
3. `air_quality` — Open-Meteo Air Quality
4. `weather` — Open-Meteo Weather
5. `mcd_burned_scar` — MODIS MCD64A1 Burned Scar (`modis-64A1-061`)

Setiap sumber disimpan sebagai dataset Parquet terpisah di `abfss://silver@georisk.dfs.core.windows.net/<source>/`.

In [ ]:
# Parameter cell (dapat di-override otomatis oleh Azure Synapse Pipeline)
storage_account = "georisk"
bronze_container = "bronze"
silver_container = "silver"
run_slot = ""  # Contoh: "2026/11/23/1200". Kosongkan untuk otomatis ambil run_slot terbaru.

In [ ]:
import json
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window

BRONZE_ROOT = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net"
SILVER_ROOT = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net"


def resolve_latest_run_slot(bronze_root: str) -> str:
    """Cari folder runs/YYYY/MM/DD/HH00 paling baru di container bronze."""
    runs_root = f"{bronze_root}/runs"
    slots = []
    for year_dir in mssparkutils.fs.ls(runs_root):
        if not year_dir.isDir:
            continue
        for month_dir in mssparkutils.fs.ls(year_dir.path):
            if not month_dir.isDir:
                continue
            for day_dir in mssparkutils.fs.ls(month_dir.path):
                if not day_dir.isDir:
                    continue
                for slot_dir in mssparkutils.fs.ls(day_dir.path):
                    if slot_dir.isDir:
                        slots.append(
                            f"{year_dir.name}/{month_dir.name}/{day_dir.name}/{slot_dir.name}"
                        )
    if not slots:
        raise RuntimeError(f"Tidak ditemukan folder run_slot di {runs_root}")
    return sorted(slots)[-1]


if not run_slot:
    run_slot = resolve_latest_run_slot(BRONZE_ROOT)

BRONZE_RUN_PATH = f"{BRONZE_ROOT}/runs/{run_slot}"
print(f"Active run_slot : {run_slot}")
print(f"Bronze run path : {BRONZE_RUN_PATH}")
print(f"Silver root     : {SILVER_ROOT}")


def path_exists(path: str) -> bool:
    try:
        return bool(mssparkutils.fs.ls(path))
    except Exception:
        return False


def concat_dedup_write_silver(
    df_new: DataFrame,
    source_name: str,
    key_cols: list[str],
    order_col: str,
    partition_cols: list[str] | None = None,
) -> int:
    """Concat batch baru dengan Silver lama, deduplikasi, lalu tulis aman via staging swap."""
    target_path = f"{SILVER_ROOT}/{source_name}"
    staging_path = f"{SILVER_ROOT}/_staging_{source_name}"

    if path_exists(target_path):
        df_existing = spark.read.parquet(target_path)
        df_combined = df_existing.unionByName(df_new, allowMissingColumns=True)
    else:
        df_combined = df_new

    window_spec = Window.partitionBy(*key_cols).orderBy(F.col(order_col).desc_nulls_last())
    df_silver = (
        df_combined.withColumn("_rn", F.row_number().over(window_spec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # Tulis ke folder staging dulu agar tidak bentrok saat overwrite path yang sama
    writer = df_silver.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(staging_path)

    if path_exists(target_path):
        mssparkutils.fs.rm(target_path, True)
    mssparkutils.fs.mv(staging_path, target_path)

    total_rows = spark.read.parquet(target_path).count()
    print(f"[{source_name}] Silver updated -> {total_rows} total rows at {target_path}")
    return total_rows


summary = {"run_slot": run_slot, "silver_counts": {}}

In [ ]:
# 1. NASA FIRMS (Hotspots)
firms_bronze = f"{BRONZE_RUN_PATH}/firms/latest_hotspots.csv"
df_firms_new = (
    spark.read.option("header", True).csv(firms_bronze)
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("brightness", F.col("brightness").cast("double"))
    .withColumn("frp", F.col("frp").cast("double"))
    .withColumn("acq_date", F.to_date("acq_date"))
    .withColumn("ingested_at_utc", F.to_timestamp("ingested_at_utc"))
    .withColumn("run_slot", F.lit(run_slot))
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
)

summary["silver_counts"]["firms"] = concat_dedup_write_silver(
    df_new=df_firms_new,
    source_name="firms",
    key_cols=["sensor", "latitude", "longitude", "acq_date", "acq_time"],
    order_col="ingested_at_utc",
    partition_cols=["acq_date"],
)

In [ ]:
# 2. MAGMA Indonesia (Aktivitas Gunung Api)
magma_bronze = f"{BRONZE_RUN_PATH}/magma/latest.json"
df_magma_raw = spark.read.option("multiLine", True).json(magma_bronze)

df_magma_new = (
    df_magma_raw.select(F.explode_outer("records").alias("rec"))
    .select(
        F.col("rec.volcano_name").alias("volcano_name"),
        F.col("rec.province").alias("province"),
        F.col("rec.activity_level").alias("activity_level"),
        F.col("rec.detail_url").alias("detail_url"),
        F.to_timestamp(F.col("rec.scraped_at_utc")).alias("scraped_at_utc"),
    )
    .filter(F.col("volcano_name").isNotNull())
    .withColumn("scraped_date", F.to_date("scraped_at_utc"))
    .withColumn("run_slot", F.lit(run_slot))
)

summary["silver_counts"]["magma"] = concat_dedup_write_silver(
    df_new=df_magma_new,
    source_name="magma",
    key_cols=["volcano_name", "activity_level", "scraped_date"],
    order_col="scraped_at_utc",
    partition_cols=["scraped_date"],
)

In [ ]:
# 3. Open-Meteo Air Quality
aq_bronze = f"{BRONZE_RUN_PATH}/air_quality/latest_air_quality.csv"
df_aq_new = (
    spark.read.option("header", True).csv(aq_bronze)
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("timestamp_utc", F.to_timestamp("timestamp_utc"))
    .withColumn("pm10", F.col("pm10").cast("double"))
    .withColumn("pm2_5", F.col("pm2_5").cast("double"))
    .withColumn("carbon_monoxide", F.col("carbon_monoxide").cast("double"))
    .withColumn("nitrogen_dioxide", F.col("nitrogen_dioxide").cast("double"))
    .withColumn("sulphur_dioxide", F.col("sulphur_dioxide").cast("double"))
    .withColumn("ozone", F.col("ozone").cast("double"))
    .withColumn("us_aqi", F.col("us_aqi").cast("double"))
    .withColumn("ingested_at_utc", F.to_timestamp("ingested_at_utc"))
    .withColumn("year", F.year("timestamp_utc"))
    .withColumn("month", F.month("timestamp_utc"))
    .withColumn("run_slot", F.lit(run_slot))
    .filter(F.col("location").isNotNull() & F.col("timestamp_utc").isNotNull())
)

summary["silver_counts"]["air_quality"] = concat_dedup_write_silver(
    df_new=df_aq_new,
    source_name="air_quality",
    key_cols=["location", "timestamp_utc"],
    order_col="ingested_at_utc",
    partition_cols=["year", "month"],
)

In [ ]:
# 4. Open-Meteo Weather
weather_bronze = f"{BRONZE_RUN_PATH}/weather/latest_weather.csv"
df_weather_new = (
    spark.read.option("header", True).csv(weather_bronze)
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("timestamp_utc", F.to_timestamp("timestamp_utc"))
    .withColumn("temperature_2m", F.col("temperature_2m").cast("double"))
    .withColumn("relative_humidity_2m", F.col("relative_humidity_2m").cast("double"))
    .withColumn("precipitation", F.col("precipitation").cast("double"))
    .withColumn("wind_speed_10m", F.col("wind_speed_10m").cast("double"))
    .withColumn("wind_direction_10m", F.col("wind_direction_10m").cast("double"))
    .withColumn("surface_pressure", F.col("surface_pressure").cast("double"))
    .withColumn("ingested_at_utc", F.to_timestamp("ingested_at_utc"))
    .withColumn("year", F.year("timestamp_utc"))
    .withColumn("month", F.month("timestamp_utc"))
    .withColumn("run_slot", F.lit(run_slot))
    .filter(F.col("location").isNotNull() & F.col("timestamp_utc").isNotNull())
)

summary["silver_counts"]["weather"] = concat_dedup_write_silver(
    df_new=df_weather_new,
    source_name="weather",
    key_cols=["location", "timestamp_utc"],
    order_col="ingested_at_utc",
    partition_cols=["year", "month"],
)

In [ ]:
# 5. MODIS MCD64A1 Burned Scar (Dataset tersendiri di Silver: concat + dedup by item_id)
mcd_bronze = f"{BRONZE_RUN_PATH}/mcd_burned_scar/latest_burned_scar.csv"
df_mcd_new = (
    spark.read.option("header", True).csv(mcd_bronze)
    .withColumn("tile_horizontal", F.col("tile_horizontal").cast("int"))
    .withColumn("tile_vertical", F.col("tile_vertical").cast("int"))
    .withColumn("bbox_west", F.col("bbox_west").cast("double"))
    .withColumn("bbox_south", F.col("bbox_south").cast("double"))
    .withColumn("bbox_east", F.col("bbox_east").cast("double"))
    .withColumn("bbox_north", F.col("bbox_north").cast("double"))
    .withColumn("start_datetime", F.to_timestamp("start_datetime"))
    .withColumn("end_datetime", F.to_timestamp("end_datetime"))
    .withColumn("created", F.to_timestamp("created"))
    .withColumn("ingested_at_utc", F.to_timestamp("ingested_at_utc"))
    .withColumn("year", F.year("start_datetime"))
    .withColumn("month", F.month("start_datetime"))
    .withColumn("run_slot", F.lit(run_slot))
    .filter(F.col("item_id").isNotNull())
)

summary["silver_counts"]["mcd_burned_scar"] = concat_dedup_write_silver(
    df_new=df_mcd_new,
    source_name="mcd_burned_scar",
    key_cols=["item_id"],
    order_col="ingested_at_utc",
    partition_cols=["year", "month"],
)

In [ ]:
# Ringkasan akhir & output untuk Synapse Pipeline
print(json.dumps(summary, indent=2))
mssparkutils.notebook.exit(json.dumps(summary))